# 跨模型结构化输出：回退（Fallback）、漂移（Drift）与 Schema 保险

无论你的主模型有多优秀，它都**一定会失败**。有时是超时，有时是幻觉，有时输出在语法上看似正确，但语义却错了。在生产环境里，你不能假装这些情况不会发生。

**每一个智能体系统都需要回退机制（fallback）。** 但问题在于：模型一旦变化，智能体的整体行为也会随之变化。因此，这件事必须在系统设计阶段就考虑，而不是等凌晨两点第一次线上故障时再补救。你不仅要测试主模型，也要测试你的“第二选择”。

本 Notebook 演示一套**三级回退策略（three-tier fallback strategy）**，并说明每一级为什么都重要：

1. **Tier 1 — 严格 JSON Schema 模式（Strict JSON Schema mode）**：由提供商约束 token 生成，使输出匹配 schema。这是最理想的情况，但并非所有模型都支持，而且它只能约束输出的*形状*，不能约束模型的*判断*。
2. **Tier 2 — Instructor + Pydantic validators**：我们通过校验、归一化和自动重试自行执行 schema。只要模型能够生成 JSON，这条路径通常就能工作。
3. **Tier 3 — 仅 Prompt + 规范化（Prompt-only + canonicalization）**：最后的兜底。我们把 schema 写进 prompt，期望模型配合，然后对返回结果做统一规范化。

我们会通过 OpenRouter 测试四个模型，让它们把同一条客户消息分诊为结构化支持工单，并展示：即使有了这三级保障，不同模型对于 priority、sentiment 这类主观字段仍然可能得出不同结论。

### 1. 严格 JSON Schema 模式约束的是形状，不是判断

所有支持 strict mode 的模型都生成了合法 JSON，但它们对**字段值本身存在分歧**。Priority 是 `p0` 还是 `p1`，可能决定是否需要立即呼叫值班人员；Sentiment 是 `angry` 还是 `frustrated`，也会改变升级路径。Schema 保证的是结构，而不是解释。在真实智能体里还有工具调用和多轮 handoff（交接），这种结构保证也只作用于单次调用，不能自动覆盖整条流水线。

### 2. 并非所有模型都支持 strict mode，因此需要 Tier 2

当主模型不可用并路由到回退模型时，回退模型未必支持 `response_format=json_schema`。如果没有 Instructor + Pydantic validators 作为第二道防线，整条流水线可能直接崩溃。Instructor 不仅提供校验，还能基于校验错误自动重试——模型会明确收到“哪里出错了”的反馈，并获得再次生成的机会。

### 3. 归一化应放在 Pydantic validators 中

把归一化映射放进 Pydantic 模型的 `field_validator`，意味着 strict、Instructor 和 degraded 三条路径都能共享同一套归一化规则。例如：`"High"` → `"p1"`、`"authentication"` → `"login"`、`"anxious"` → `"frustrated"`。这不是可有可无的补丁，而是 schema insurance（Schema 保险）。

### 4. 要用完整流水线测试回退模型

不要只测试“这个模型能不能生成文本”。真正要验证的是：它的输出能否完整通过三级回退 → 解析 → 归一化 → 校验 → handoff 这一整条链路？只要其中任何一步失败，这个回退模型就没有实际价值。

### 5. 回退层级越深，延迟通常越高

Tier 1（strict）通常最快：一次 API 调用，不需要重试。Tier 2（Instructor）在校验失败时可能发生重试。Tier 3（degraded + canonicalize）需要额外 API 调用和后处理。因此，这些成本必须计入 SLA 与 timeout budget（超时预算）。

### 6. 不同模型拥有不同的“判断尺度”

这一点很细，但非常关键：面对同一个愤怒客户，两个模型可能分别判定为 `p0/angry` 和 `p1/frustrated`。如果你的路由逻辑、SLA 计时器或升级规则依赖这些字段，**你就必须知道回退链中的每个模型会怎样给这些字段打标签**，并判断这种差异是否能被业务接受。这也是为什么回退模型应当在 Hardening（加固）阶段就纳入测试，而不是等第一次生产故障之后再看。

In [1]:
!pip install instructor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 22.0 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.13.0
    Uninstalling jiter-0.13.0:
      Successfully uninstalled jiter-0.13.0


In [2]:
import json, re, time, random, requests
from typing import Any, Dict, List, Literal
from pydantic import BaseModel, field_validator
import os
from dotenv import load_dotenv
import instructor
from openai import OpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
BASE = "https://openrouter.ai/api/v1"

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 归一化映射（Normalization Maps）

这些映射是我们抵御 model drift（模型漂移）的核心防线。不同模型可能会用不同的值表达同一个语义概念——例如用 `"High"` 代替 `"p1"`，用 `"authentication"` 代替 `"login"`，用 `"anxious"` 代替 `"frustrated"`。

我们在前面就定义这些映射，是因为 Pydantic validators（Tier 2 中由 Instructor 使用）和最后兜底的 `canonicalize()` 函数（Tier 3）都依赖它们。

In [3]:
CATEGORY_MAP = {
    "authentication": "login",
    "authentication/login": "login",
    "account access": "login",
    "login": "login",
    "billing": "billing",
    "bug": "bug",
    "performance": "performance",
    "feature_request": "feature_request",
    "other": "other",
}

PRIORITY_MAP = {
    "p0": "p0", "p1": "p1", "p2": "p2", "p3": "p3",
    "critical": "p0",
    "urgent": "p1",
    "high": "p1",
    "medium": "p2",
    "low": "p3",
}

SENTIMENT_MAP = {
    "calm": "calm",
    "frustrated": "frustrated",
    "angry": "angry",
    "frustrated/urgent": "frustrated",
    "urgent": "frustrated",
    "anxious": "frustrated",
}

## 目标 Schema 与 Pydantic 模型

这里有两个不同的东西，它们分别承担不同职责：

1. **JSON Schema** 会在严格模式（Tier 1）下发送给 API 提供商。提供商利用它在 token 层执行 constrained decoding（约束解码）。
2. 带有 `Literal` 类型和 `field_validator` 的 **Pydantic 模型** 是我们的*本地执行层*。它利用上面的映射表归一化输入值，并拒绝无法映射的内容。Instructor（Tier 2）使用这个模型校验响应，并在失败时重试。

注意这些 `field_validator`：当模型返回 `"priority": "High"` 时，validator 会通过 `PRIORITY_MAP` 将其映射为 `"p1"`。如果某个值完全无法映射，validator 会抛出 `ValueError`；Instructor 会捕获这个错误，把错误信息注入 prompt，并明确告诉模型哪里出了问题，让它再尝试一次。

In [4]:
SUPPORT_TICKET_SCHEMA = {
    "type": "object",
    "properties": {
        "ticket_id": {"type": "string"},
        "summary": {"type": "string"},
        "category": {"type": "string", "enum": ["billing","login","bug","feature_request","performance","other"]},
        "priority": {"type": "string", "enum": ["p0","p1","p2","p3"]},
        "customer_sentiment": {"type": "string", "enum": ["calm","frustrated","angry"]},
        "repro_steps": {"type": "array", "items": {"type": "string"}},
        "expected_behavior": {"type": "string"},
        "actual_behavior": {"type": "string"},
        "suggested_next_action": {"type": "string"},
    },
    "required": [
        "ticket_id","summary","category","priority","customer_sentiment",
        "repro_steps","expected_behavior","actual_behavior","suggested_next_action"
    ],
    "additionalProperties": False,
}


class SupportTicket(BaseModel):
    ticket_id: str
    summary: str
    category: Literal["billing", "login", "bug", "feature_request", "performance", "other"]
    priority: Literal["p0", "p1", "p2", "p3"]
    customer_sentiment: Literal["calm", "frustrated", "angry"]
    repro_steps: List[str]
    expected_behavior: str
    actual_behavior: str
    suggested_next_action: str

    @field_validator("category", mode="before")
    @classmethod
    def normalize_category(cls, v: str) -> str:
        return CATEGORY_MAP.get(str(v).strip().lower(), "other")

    @field_validator("priority", mode="before")
    @classmethod
    def normalize_priority(cls, v: str) -> str:
        mapped = PRIORITY_MAP.get(str(v).strip().lower())
        if mapped:
            return mapped
        raise ValueError(f"Cannot map priority '{v}' to p0/p1/p2/p3. Valid inputs: {list(PRIORITY_MAP.keys())}")

    @field_validator("customer_sentiment", mode="before")
    @classmethod
    def normalize_sentiment(cls, v: str) -> str:
        return SENTIMENT_MAP.get(str(v).strip().lower(), "frustrated")

## Instructor Client 配置

我们使用 [Instructor](https://github.com/instructor-ai/instructor) 包装 OpenAI client，从而获得经过 Pydantic 校验的 structured output（结构化输出）以及自动重试能力。当模型返回 validator 无法归一化的值（例如 `"priority": "ASAP"`）时，validator 会抛错，Instructor 会把错误注入 prompt，并让模型在明确知道问题所在的情况下再尝试一次。

这就是 **Tier 2**：它比 prompt engineering（Tier 3）更强制，又比 strict schema mode（Tier 1）更通用。只要模型能够生成类似 JSON 的输出，这一层通常都能工作。

In [5]:
instructor_client = instructor.from_openai(
    OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    ),
    mode=instructor.Mode.JSON,  # 适用于任何能够生成 JSON 的模型
)

## 客户消息（The Customer Message）

我们使用一张任何模型都应该能够完成分诊的支持工单。客户明显感到沮丧，同时有很紧急的截止时间（一小时后要做 demo），而且已经尝试过自助解决（两次重置密码）。接下来看看不同模型如何解读完全相同的情境。

In [6]:
PROMPT = """Turn this into a structured support ticket.

Customer message:
"I cannot log in since yesterday. I reset my password twice and it still says 'invalid credentials'.
I have a demo in one hour. This is ridiculous. Please fix this now."
"""

## API 辅助函数（API Helper）

`call_openrouter` 会在遇到 5xx 错误时使用带 jitter（抖动）的 backoff（退避）策略进行重试；对于 4xx（错误请求、认证失败）则立即抛出异常。Tier 1 的 strict mode 和 Tier 3 的 degraded mode 会使用它，而 Tier 2 使用 Instructor client。

In [7]:
def call_openrouter(payload: Dict[str, Any], retries: int = 2, timeout_s: int = 25) -> Dict[str, Any]:
    last = None
    for _ in range(retries):
        r = requests.post(
            f"{BASE}/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
            },
            json=payload,
            timeout=timeout_s,
        )

        if 500 <= r.status_code <= 599:
            last = f"{r.status_code}: {r.text[:240]}"
            time.sleep(0.4 + random.random() * 0.6)
            continue

        if r.status_code >= 400:
            raise RuntimeError(f"{r.status_code}: {r.text[:600]}")

        return r.json()

    raise RuntimeError(f"OpenRouter failed after retries. Last={last}")

## 三级回退策略（Three-Tier Fallback Strategy）

### Tier 1：严格 JSON Schema 模式（`response_format=json_schema`）

这是当前可用的最强方案。我们把完整 JSON schema 直接发送给 API，并设置 `"strict": True`，让提供商在 token 生成层执行约束。

但要注意，strict mode 只能约束**单次 LLM completion 调用**——它会迫使模型这一轮直接生成符合 schema 的 token。在生产级 agentic system（智能体系统）中，下面几类场景仍可能让保证失效：

- **中间存在 tool call（工具调用）。** 模型先生成工具调用，拿到结果，再生成最终结构化输出。工具结果可能注入意外内容，从而改变模型对 schema 字段值的“理解”。
- **多轮 handoff（交接）。** 如果 Model A 的结构化输出进入 Model B 的 prompt（甚至只是进入 Model A 的下一轮），schema enforcement 只作用于每一次单独的生成边界，而不会自动贯穿整条链路。
- **Provider-level quirks（提供商层差异）。** OpenRouter 实际代理到底层提供商。所谓 `strict` 是否真正有效，取决于底层提供商是否真的实现 constrained decoding，而不只是声称支持。
- **字段值仍会漂移。** 即便 strict mode 工作正常，回退模型仍可能对同一工单做出不同判断——给出不同优先级、识别出不同情绪。

因此，strict mode 能保证一次调用得到格式良好的 JSON，却**不能**保证不同模型、不同轮次以及整条流水线上的*判断一致性*。

### Tier 2：Instructor + Pydantic Validators

当 strict mode 失败时，我们改用 Instructor 配合 Pydantic 模型。Instructor 要求模型返回 JSON，再用 `SupportTicket` 校验响应（其中包括负责归一化的 `field_validator`）；如果任一步失败，它会**把校验错误注入 prompt 并自动重试**。这比单纯 prompt engineering 更强，因为模型能够得到明确的错误反馈。

### Tier 3：仅 Prompt + Canonicalization（最后兜底）

如果连 Instructor 都拿不到合法输出——比如模型完全不支持 JSON mode，或者所有重试都失败——就退回普通 prompt engineering。我们在 system message 里描述 schema 约束，解析模型返回的任何内容，然后由 `canonicalize()` 统一 key、映射值并补充默认字段。这条路径最弱，但至少比直接崩溃更好。

In [8]:
STRICT_SYSTEM = "Return ONLY valid JSON. No markdown. No code fences."

def run_strict(model: str) -> Dict[str, Any]:
    """Tier 1：通过提供商执行严格 JSON Schema 模式。"""
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": STRICT_SYSTEM},
            {"role": "user", "content": PROMPT},
        ],
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "support_ticket", "strict": True, "schema": SUPPORT_TICKET_SCHEMA},
        },
    }
    t0 = time.time()
    data = call_openrouter(payload)
    latency = time.time() - t0
    content = data["choices"][0]["message"]["content"]
    return {"model": model, "mode": "strict", "latency_s": round(latency, 3), "raw_content": content}

### Tier 2：由 Instructor 驱动的结构化输出

这里我们从“希望模型配合”升级为真正的**强制执行**。Instructor 会把 Pydantic schema 作为请求的一部分发送出去，解析响应，执行 `field_validator`（利用我们的映射表归一化字段值）；如果任一步失败，就把校验错误注入对话并重试。

`max_retries=2` 表示最多 3 次尝试（首次 + 2 次重试）。如果模型持续失败，就继续下落到 Tier 3。

In [9]:
def run_instructor(model: str) -> Dict[str, Any]:
    """Tier 2：Instructor + Pydantic 校验，并在失败时自动重试。"""
    t0 = time.time()
    ticket = instructor_client.chat.completions.create(
        model=model,
        response_model=SupportTicket,
        max_retries=2,
        messages=[
            {"role": "system", "content": "You are a customer support triage agent. Return a structured support ticket."},
            {"role": "user", "content": PROMPT},
        ],
    )
    latency = time.time() - t0
    return {
        "model": model,
        "mode": "instructor",
        "latency_s": round(latency, 3),
        "ticket": ticket,
        "raw_content": ticket.model_dump_json(indent=2),
    }

### Tier 3：仅 Prompt 的 JSON（最后兜底）

如果连 Instructor 都无法得到合法输出——比如模型完全不支持 JSON mode，或者耗尽全部重试——我们就退回 **prompt engineering（提示工程）**。在 system message 中描述 schema 约束，并要求模型返回纯 JSON。

这天然是最不可靠的一条路径。模型可能会：
- 把 JSON 包在 ```json code fence（代码围栏）里
- 返回嵌套对象，而不是扁平结构
- 使用不同 key 名（例如 `subject` 而不是 `summary`）
- 选择允许枚举之外的值（例如 `"High"` 而不是 `"p1"`）

因此，这条路径最终会进入 `canonicalize()`——也就是紧急归一化器。

In [10]:
DEGRADED_SYSTEM = """Return ONLY valid JSON. No markdown. No code fences.
Return a SINGLE flat JSON object with exactly these keys:
ticket_id, summary, category, priority, customer_sentiment, repro_steps, expected_behavior, actual_behavior, suggested_next_action.

Rules:
- category must be one of: billing, login, bug, feature_request, performance, other
- priority must be one of: p0, p1, p2, p3
- customer_sentiment must be one of: calm, frustrated, angry
- repro_steps must be a JSON array of strings (even if empty)
- Do not add any other keys
"""

def run_degraded(model: str) -> Dict[str, Any]:
    """Tier 3：仅靠 Prompt 生成 JSON——最后的兜底路径。"""
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": DEGRADED_SYSTEM},
            {"role": "user", "content": PROMPT},
        ],
    }
    t0 = time.time()
    data = call_openrouter(payload)
    latency = time.time() - t0
    content = data["choices"][0]["message"]["content"]
    return {"model": model, "mode": "degraded", "latency_s": round(latency, 3), "raw_content": content}

## 健壮的 JSON 解析（Tier 3）

处于 degraded mode（降级模式）时，模型经常会把 JSON 包在 ```json code fence 中，在 JSON 前后添加说明文字，或者返回嵌套对象。我们必须优雅处理这些情况——因为在生产环境里，你不能假设每次失败后都有机会重新 prompt。

In [11]:
def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\n?", "", s)
        s = re.sub(r"\n?```$", "", s)
    return s.strip()

def extract_first_json_object(text: str) -> str:
    s = text.strip()
    start = s.find("{")
    if start == -1:
        raise ValueError("No JSON object start found")
    depth = 0
    for i in range(start, len(s)):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                return s[start:i+1]
    raise ValueError("No complete JSON object found")

def parse_json_robust(content: str) -> Dict[str, Any]:
    cleaned = strip_code_fences(content)
    try:
        return json.loads(cleaned)
    except Exception:
        return json.loads(extract_first_json_object(cleaned))

## 最后的规范化兜底（Last-Resort Canonicalization，Tier 3）

`canonicalize()` 是模型输出的“急诊室”。只有 Tier 1（strict）和 Tier 2（Instructor）都失败时才会进入这里，这意味着当前输出大概率已经很混乱。它会：

- 接受替代 key 名（`subject` → `summary`、`steps_taken` → `repro_steps`）
- 使用归一化映射处理非标准值
- 当前面方法都无效时，根据上下文推断 priority
- 为缺失字段提供合理默认值

注意，上面的归一化映射不仅供这里使用，Tier 2 的 Pydantic validators 也共享同一套规则。这个函数是 validators 也帮不上忙时的最后 backstop（兜底层）——通常意味着模型返回的内容已经严重偏离 schema，连 Instructor 的重试都无法修复。

In [12]:
def to_list(x: Any) -> List[str]:
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if "\n" in s:
            parts = [p.strip(" -\t") for p in s.split("\n")]
            return [p for p in parts if p]
        return [s]
    return []

def infer_priority(original_prompt: str, summary: str, sentiment: str, category: str) -> str:
    """当模型没有给出可用结果时，用启发式规则推断 priority。"""
    t = (original_prompt + " " + (summary or "")).lower()
    s = (sentiment or "").lower()
    cat = (category or "").lower()

    urgent_signal = ("demo" in t and ("1 hour" in t or "one hour" in t)) or "urgent" in t or "immediate" in t
    blocked_login = (cat == "login") and ("cannot log in" in t or "unable to log in" in t)

    if blocked_login and ("angry" in s or urgent_signal):
        return "p0"
    if urgent_signal:
        return "p1"
    if blocked_login:
        return "p1"
    return "p2"

def canonicalize(raw: Dict[str, Any]) -> Dict[str, Any]:
    """Tier 3 输出的紧急归一化器：映射替代 key 与非标准值。"""
    ticket_id = str(raw.get("ticket_id") or raw.get("id") or "TCK-0000").strip()
    summary = str(raw.get("summary") or raw.get("subject") or raw.get("title") or "").strip()

    category_raw = str(raw.get("category") or "other").strip().lower()
    category = CATEGORY_MAP.get(category_raw, "other")

    sentiment_raw = str(raw.get("customer_sentiment") or raw.get("sentiment") or "frustrated").strip().lower()
    sentiment = SENTIMENT_MAP.get(sentiment_raw, "frustrated")

    priority_raw = str(raw.get("priority") or "").strip().lower()
    priority = PRIORITY_MAP.get(priority_raw, None)
    if priority is None:
        priority = infer_priority(PROMPT, summary, sentiment, category)

    repro_steps = to_list(raw.get("repro_steps") or raw.get("steps_taken") or raw.get("actions_taken"))
    expected_behavior = str(raw.get("expected_behavior") or "").strip()
    actual_behavior = str(raw.get("actual_behavior") or raw.get("current_behavior") or "").strip()
    next_action = str(raw.get("suggested_next_action") or raw.get("requested_action") or "").strip()

    return {
        "ticket_id": ticket_id if ticket_id else "TCK-0000",
        "summary": summary if summary else "Customer cannot access account",
        "category": category,
        "priority": priority,
        "customer_sentiment": sentiment,
        "repro_steps": repro_steps if repro_steps else ["Attempt login", "Reset password", "Attempt login again"],
        "expected_behavior": expected_behavior if expected_behavior else "User should be able to log in successfully.",
        "actual_behavior": actual_behavior if actual_behavior else "User receives invalid credentials error after password reset.",
        "suggested_next_action": next_action if next_action else "Check auth logs and account lockout status; escalate if needed.",
    }

## 完整的三级回退流水线

`run_model()` 实现完整策略：

1. **Tier 1 — 尝试 strict mode** → 解析 → 用 Pydantic 校验（validators 同时归一化）
2. **Tier 2 — 尝试 Instructor** → Pydantic 校验 + 归一化 → 失败时自动重试
3. **Tier 3 — 仅 Prompt** → 健壮解析 → `canonicalize()` → 再用 Pydantic 校验

层级越往后，可靠性越弱，但兼容性越广。在生产环境中，大多数调用理应在 Tier 1 成功。如果主模型宕机，而回退模型不支持 strict mode，Tier 2 会接住；如果连这一层也失败，Tier 3 就是紧急通道。

问题从来不是主模型*会不会*失败，而是它*什么时候*失败，以及系统在那时能够优雅降级（graceful degradation）还是直接崩溃。

In [13]:
def run_model(model: str) -> Dict[str, Any]:
    strict_failure = None
    instructor_failure = None

    # ── Tier 1：严格 JSON Schema 模式 ───────────────────────────────
    try:
        out = run_strict(model)
        raw = parse_json_robust(out["raw_content"])
        ticket = SupportTicket.model_validate(raw)

        return {
            "model": model,
            "mode": "strict",
            "tier": 1,
            "strict_failed": False,
            "instructor_failed": False,
            "ok": True,
            "latency_s": out["latency_s"],
            "priority": ticket.priority,
            "category": ticket.category,
            "sentiment": ticket.customer_sentiment,
            "n_repro_steps": len(ticket.repro_steps),
            "strict_failure": None,
            "instructor_failure": None,
            "note": "Tier 1: Strict JSON Schema succeeded.",
            "raw_excerpt": out["raw_content"][:300],
        }
    except Exception as e_strict:
        strict_failure = f"{type(e_strict).__name__}: {str(e_strict)[:220]}"

    # ── Tier 2：Instructor + Pydantic 校验 ────────────────────────────
    try:
        out2 = run_instructor(model)
        ticket2 = out2["ticket"]

        return {
            "model": model,
            "mode": "instructor",
            "tier": 2,
            "strict_failed": True,
            "instructor_failed": False,
            "ok": True,
            "latency_s": out2["latency_s"],
            "priority": ticket2.priority,
            "category": ticket2.category,
            "sentiment": ticket2.customer_sentiment,
            "n_repro_steps": len(ticket2.repro_steps),
            "strict_failure": strict_failure,
            "instructor_failure": None,
            "note": "Tier 2: Strict failed \u2192 Instructor + Pydantic validators succeeded.",
            "raw_excerpt": out2["raw_content"][:300],
        }
    except Exception as e_instr:
        instructor_failure = f"{type(e_instr).__name__}: {str(e_instr)[:220]}"

    # ── Tier 3：仅 Prompt + canonicalize ──────────────────────────────
    try:
        out3 = run_degraded(model)
        raw3 = parse_json_robust(out3["raw_content"])
        canonical = canonicalize(raw3)
        ticket3 = SupportTicket.model_validate(canonical)

        return {
            "model": model,
            "mode": "degraded_canonicalized",
            "tier": 3,
            "strict_failed": True,
            "instructor_failed": True,
            "ok": True,
            "latency_s": out3["latency_s"],
            "priority": ticket3.priority,
            "category": ticket3.category,
            "sentiment": ticket3.customer_sentiment,
            "n_repro_steps": len(ticket3.repro_steps),
            "strict_failure": strict_failure,
            "instructor_failure": instructor_failure,
            "note": "Tier 3: Strict + Instructor failed \u2192 prompt-only + canonicalize \u2192 validated.",
            "raw_excerpt": out3["raw_content"][:300],
        }
    except Exception as e_degraded:
        return {
            "model": model,
            "mode": "all_failed",
            "tier": 0,
            "strict_failed": True,
            "instructor_failed": True,
            "ok": False,
            "latency_s": 0,
            "priority": "unknown",
            "category": "unknown",
            "sentiment": "unknown",
            "n_repro_steps": 0,
            "strict_failure": strict_failure,
            "instructor_failure": instructor_failure,
            "note": f"All tiers failed. Last error: {str(e_degraded)[:200]}",
            "raw_excerpt": "",
        }

## 运行全部 4 个模型 × 2 次试验

我们让每个模型针对同一输入运行**两次**，检查三个方面：
- **跨模型漂移（Cross-model drift）**：不同模型是否会给出不同的 priority / sentiment？
- **逐次运行稳定性（Run-to-run stability）**：同一个模型连续两次是否给出相同答案？
- **层级分布（Tier distribution）**：哪些模型能在 Tier 1 成功，哪些必须落到 Tier 2 或 Tier 3？

In [14]:
MODELS = [
    "openai/gpt-5.2",
    "qwen/qwen3-max-thinking",
    "anthropic/claude-haiku-4.5",
    "minimax/minimax-m2.5",
]

results = []
for m in MODELS:
    for trial in range(2):
        print(f"  Running {m} (trial {trial+1}/2)...")
        results.append(run_model(m))

print(f"\nDone: {len(results)} runs completed.")

  Running openai/gpt-5.2 (trial 1/2)...
  Running openai/gpt-5.2 (trial 2/2)...
  Running qwen/qwen3-max-thinking (trial 1/2)...
  Running qwen/qwen3-max-thinking (trial 2/2)...
  Running anthropic/claude-haiku-4.5 (trial 1/2)...
  Running anthropic/claude-haiku-4.5 (trial 2/2)...
  Running minimax/minimax-m2.5 (trial 1/2)...
  Running minimax/minimax-m2.5 (trial 2/2)...

Done: 8 runs completed.


## 并排对比：每个模型到底产出了什么？

这里开始进入真正影响生产行为的部分。重点关注：
- **Tier（回退层级）**：每个模型最终落在哪一级——它直接告诉你前面的哪一层失效了
- **Priority（优先级）**：`p0` 还是 `p1`——这可能决定凌晨 2 点是否需要立刻呼叫值班人员，还是工单可以等到早晨
- **Sentiment（情绪）**：`angry` 还是 `frustrated`——它会改变自动回复语气和升级路径
- **Repro steps count（复现步骤数量）**：模型从同一条消息里提取出了多少细节
- **Latency（延迟）**：一路下落到更低回退层级时，真实需要付出的时间成本

In [15]:
# ── Per-run detail table ──────────────────────────────────────────────
print(f"{'Model':<30} {'#':>2} {'Tier':>4} {'Mode':<25} {'Pri':>3} {'Sentiment':<12} {'Steps':>5} {'Latency':>8}")
print("\u2500" * 95)
for i, r in enumerate(results):
    trial = (i % 2) + 1
    print(f"{r['model']:<30} {trial:>2} {r['tier']:>4} {r['mode']:<25} {r['priority']:>3} {r['sentiment']:<12} {r['n_repro_steps']:>5} {r['latency_s']:>7.1f}s")

# ── Highlight cross-model disagreements ───────────────────────────────
priorities = {}
sentiments = {}
tiers = {}
for r in results:
    short = r["model"].split("/")[-1]
    priorities.setdefault(short, set()).add(r["priority"])
    sentiments.setdefault(short, set()).add(r["sentiment"])
    tiers.setdefault(short, set()).add(r["tier"])

print()
print("=" * 70)
print(" CROSS-MODEL DRIFT: Same customer, same schema, different judgments")
print("=" * 70)

print()
print("Priority assignments (p0 = page someone, p1 = next morning):")
for m, ps in priorities.items():
    marker = "  \u2713" if len(ps) == 1 else "  \u26a0 UNSTABLE"
    print(f"  {m:<28} \u2192 {', '.join(sorted(ps))}{marker}")

print()
print("Sentiment readings (angry = escalate, frustrated = empathize):")
for m, ss in sentiments.items():
    marker = "  \u2713" if len(ss) == 1 else "  \u26a0 UNSTABLE"
    print(f"  {m:<28} \u2192 {', '.join(sorted(ss))}{marker}")

print()
print("Tier distribution (which fallback level did each model land on?):")
for m, ts in tiers.items():
    tier_str = ", ".join(f"Tier {t}" for t in sorted(ts))
    print(f"  {m:<28} \u2192 {tier_str}")

# ── Strict + Instructor failures ─────────────────────────────────────
strict_failures = [r for r in results if r["strict_failed"]]
instructor_failures = [r for r in results if r.get("instructor_failed")]
if strict_failures:
    print()
    print("=" * 70)
    print(f" TIER 1 FAILURES: {len(strict_failures)} of {len(results)} runs failed strict mode")
    print("=" * 70)
    for r in strict_failures:
        print(f"  {r['model']}")
        print(f"    Strict error: {(r['strict_failure'] or '')[:120]}")
        if r.get("instructor_failed"):
            print(f"    Instructor error: {(r.get('instructor_failure') or '')[:120]}")
            print(f"    Rescued by: Tier 3 (prompt-only + canonicalize)")
        else:
            print(f"    Rescued by: Tier 2 (Instructor + Pydantic)")

# ── Latency comparison by tier ────────────────────────────────────────
print()
print("=" * 70)
print(" LATENCY BY TIER")
print("=" * 70)
for tier_num, tier_name in [(1, "Strict"), (2, "Instructor"), (3, "Degraded+canonicalize")]:
    lats = [r["latency_s"] for r in results if r["tier"] == tier_num]
    if lats:
        print(f"  Tier {tier_num} ({tier_name}): avg {sum(lats)/len(lats):.1f}s  (n={len(lats)})")

Model                           # Tier Mode                      Pri Sentiment    Steps  Latency
───────────────────────────────────────────────────────────────────────────────────────────────
openai/gpt-5.2                  1    1 strict                     p0 angry            7     6.2s
openai/gpt-5.2                  2    1 strict                     p0 angry            7     6.9s
qwen/qwen3-max-thinking         1    1 strict                     p1 frustrated       4     6.2s
qwen/qwen3-max-thinking         2    1 strict                     p1 frustrated       4     6.0s
anthropic/claude-haiku-4.5      1    1 strict                     p0 angry            7     7.0s
anthropic/claude-haiku-4.5      2    1 strict                     p0 angry            5     5.8s
minimax/minimax-m2.5            1    1 strict                     p1 frustrated       6     5.5s
minimax/minimax-m2.5            2    1 strict                     p0 frustrated       3     2.6s

 CROSS-MODEL DRIFT: Same custo

## 原始输出样例（Raw Output Samples）

下面直接看看模型实际返回了什么。这样可以把 drift（漂移）从抽象概念变成具体差异：你能清楚看到不同模型如何用不同方式组织同一份信息，以及最终由哪一级 fallback（回退）接住它们。

In [16]:
seen_models = []
for r in results:
    if r["model"] not in seen_models:
        seen_models.append(r["model"])

for model_name in seen_models:
    model_runs = [r for r in results if r["model"] == model_name]
    short = model_name.split("/")[-1]
    r0 = model_runs[0]
    print(f"{'\u2501' * 80}")
    print(f"  {short}  |  tier: {r0['tier']}  |  mode: {r0['mode']}  |  priority: {r0['priority']}  |  sentiment: {r0['sentiment']}")
    print(f"{'\u2501' * 80}")

    # Show key fields from the raw output
    raw = r0["raw_excerpt"]
    try:
        # Try to parse (close truncated JSON if needed)
        parsed = json.loads(raw + ("}" * max(0, raw.count("{") - raw.count("}"))))
    except Exception:
        parsed = None

    if parsed:
        for key in ["ticket_id", "summary", "category", "priority", "customer_sentiment", "repro_steps"]:
            if key in parsed:
                val = parsed[key]
                if isinstance(val, list):
                    print(f"  {key}: [{len(val)} items] {', '.join(str(v)[:40] for v in val[:3])}{'...' if len(val) > 3 else ''}")
                else:
                    print(f"  {key}: {val}")
    else:
        print(f"  {raw[:300]}")

    # Show if the two trials agreed
    if len(model_runs) > 1:
        r1 = model_runs[1]
        diffs = []
        for field in ["priority", "sentiment", "n_repro_steps", "tier"]:
            if r0.get(field) != r1.get(field):
                diffs.append(f"{field}: {r0.get(field)} vs {r1.get(field)}")
        if diffs:
            print(f"  \u26a0 Trial 1 vs 2 differ: {'; '.join(diffs)}")
        else:
            print(f"  \u2713 Both trials agree on priority, sentiment, steps, tier")
    print()

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  gpt-5.2  |  tier: 1  |  mode: strict  |  priority: p0  |  sentiment: angry
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  {"ticket_id":"TCK-20260326-0001","summary":"User cannot log in; password reset twice but still gets 'invalid credentials' error; urgent due to demo in 1 hour","category":"login","priority":"p0","customer_sentiment":"angry","repro_steps":["Go to login page","Enter username/email and password","Submit
  ✓ Both trials agree on priority, sentiment, steps, tier

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  qwen3-max-thinking  |  tier: 1  |  mode: strict  |  priority: p1  |  sentiment: frustrated
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  {
  "ticket_id": "TKT-20231005-001",
  "summary": "Unable to log in after multiple password resets",
  "category": "login",
  "priority": "p1"

## 量化漂移分析（Quantified Drift Analysis）

接下来把模型漂移真正量化。这里的 **drift score（漂移分数）** 由三部分组成：
- **Priority drift（优先级漂移）**：同一模型在两次试验中给出不同 priority 时记 1 分
- **Steps drift（步骤漂移）**：复现步骤数量变化的归一化幅度
- **Tier penalty（层级惩罚）**：Tier 1 为 0.0，Tier 2 为 0.25，Tier 3 为 0.5

漂移分数为 0，表示模型在 Tier 1 上完全稳定；分数越高，说明它的行为越难预测。

In [17]:
import pandas as pd

df = pd.DataFrame(results).copy()

# Add trial index per model
df["trial"] = df.groupby("model").cumcount() + 1

# ── Per-run summary table ────────────────────────────────────────────
summary_cols = ["model", "trial", "tier", "mode", "latency_s",
                "priority", "category", "sentiment", "n_repro_steps", "note"]
print("Per-Run Results")
print("=" * 40)
display(df[summary_cols])

Per-Run Results


,model,trial,tier,mode,latency_s,priority,category,sentiment,n_repro_steps,note
0,openai/gpt-5.2,1,1,strict,6.193,p0,login,angry,7,Tier 1: Strict JSON Schema succeeded.
1,openai/gpt-5.2,2,1,strict,6.881,p0,login,angry,7,Tier 1: Strict JSON Schema succeeded.
2,qwen/qwen3-max-thinking,1,1,strict,6.182,p1,login,frustrated,4,Tier 1: Strict JSON Schema succeeded.
3,qwen/qwen3-max-thinking,2,1,strict,6.003,p1,login,frustrated,4,Tier 1: Strict JSON Schema succeeded.
4,anthropic/claude-haiku-4.5,1,1,strict,7.000,p0,login,angry,7,Tier 1: Strict JSON Schema succeeded.
5,anthropic/claude-haiku-4.5,2,1,strict,5.772,p0,login,angry,5,Tier 1: Strict JSON Schema succeeded.
6,minimax/minimax-m2.5,1,1,strict,5.471,p1,login,frustrated,6,Tier 1: Strict JSON Schema succeeded.
7,minimax/minimax-m2.5,2,1,strict,2.600,p0,login,frustrated,3,Tier 1: Strict JSON Schema succeeded.


In [18]:
# ── Per-model delta: what changed between trials? ─────────────────────
def delta_summary(g: pd.DataFrame) -> pd.Series:
    diffs = []
    for field in ["priority", "category", "sentiment", "n_repro_steps", "mode", "tier"]:
        if g[field].nunique(dropna=True) > 1:
            diffs.append(field)

    change = ("Changed across trials: " + ", ".join(diffs) + ".") if diffs else "Stable across both trials."

    return pd.Series({
        "trials": len(g),
        "changed_fields": ", ".join(diffs) if diffs else "none",
        "delta_note": change,
        "latency_min": round(float(g["latency_s"].min()), 3),
        "latency_max": round(float(g["latency_s"].max()), 3),
    })

delta = df.groupby("model").apply(delta_summary, include_groups=False).reset_index()
print("\nRun-to-Run Stability (same model, same input)")
print("=" * 50)
display(delta)

# ── Drift scoring (updated for three tiers) ──────────────────────────
priority_rank = {"p0": 3, "p1": 2, "p2": 1, "p3": 0}
tier_penalty_map = {1: 0.0, 2: 0.25, 3: 0.5, 0: 1.0}

def drift_for_model(g: pd.DataFrame) -> pd.Series:
    pr = g["priority"].map(priority_rank)
    priority_drift = int(pr.nunique(dropna=True) > 1)

    steps = pd.to_numeric(g["n_repro_steps"], errors="coerce")
    steps_min = steps.min() if steps.notna().any() else None
    steps_max = steps.max() if steps.notna().any() else None
    steps_drift = float(steps_max - steps_min) if steps_min is not None else 0.0
    steps_drift_norm = min(1.0, steps_drift / 10.0)

    # Tier penalty: worst tier used across trials
    worst_tier = int(g["tier"].max()) if g["tier"].notna().any() else 0
    tier_penalty = tier_penalty_map.get(worst_tier, 0.5)

    drift_score = priority_drift + steps_drift_norm + tier_penalty

    return pd.Series({
        "trials": len(g),
        "tiers_used": ", ".join(f"T{int(t)}" for t in sorted(g["tier"].dropna().unique())),
        "worst_tier": worst_tier,
        "priority_values": ", ".join(sorted(g["priority"].dropna().unique().tolist())),
        "steps_min": int(steps_min) if steps_min is not None else None,
        "steps_max": int(steps_max) if steps_max is not None else None,
        "priority_drift": priority_drift,
        "steps_drift": round(steps_drift, 2),
        "tier_penalty": tier_penalty,
        "drift_score": round(drift_score, 2),
        "avg_latency_s": round(float(g["latency_s"].mean()), 3),
        "p95_latency_s": round(float(g["latency_s"].quantile(0.95)), 3),
    })

drift = df.groupby("model").apply(drift_for_model, include_groups=False).reset_index()
print("\nDrift Score by Model (lower = more stable)")
print("=" * 50)
display(drift)


Run-to-Run Stability (same model, same input)


,model,trials,changed_fields,delta_note,latency_min,latency_max
0,anthropic/claude-haiku-4.5,2,n_repro_steps,Changed across trials: n_repro_steps.,5.772,7.000
1,minimax/minimax-m2.5,2,"priority, n_repro_steps","Changed across trials: priority, n_repro_steps.",2.600,5.471
2,openai/gpt-5.2,2,none,Stable across both trials.,6.193,6.881
3,qwen/qwen3-max-thinking,2,none,Stable across both trials.,6.003,6.182



Drift Score by Model (lower = more stable)


,model,trials,tiers_used,worst_tier,priority_values,steps_min,steps_max,priority_drift,steps_drift,tier_penalty,drift_score,avg_latency_s,p95_latency_s
0,anthropic/claude-haiku-4.5,2,T1,1,p0,5,7,0,2.0,0.0,0.2,6.386,6.939
1,minimax/minimax-m2.5,2,T1,1,"p0, p1",3,6,1,3.0,0.0,1.3,4.035,5.327
2,openai/gpt-5.2,2,T1,1,p0,7,7,0,0.0,0.0,0.0,6.537,6.847
3,qwen/qwen3-max-thinking,2,T1,1,p1,4,4,0,0.0,0.0,0.0,6.093,6.173


In [19]:
# ── Overall summary ──────────────────────────────────────────────────
tier_counts = df["tier"].value_counts().to_dict()
overall = {
    "models_tested": int(df["model"].nunique()),
    "total_runs": int(len(df)),
    "tier_1_runs": int(tier_counts.get(1, 0)),
    "tier_2_runs": int(tier_counts.get(2, 0)),
    "tier_3_runs": int(tier_counts.get(3, 0)),
    "all_failed_runs": int(tier_counts.get(0, 0)),
    "avg_latency_s": round(float(df["latency_s"].mean()), 3),
    "p95_latency_s": round(float(df["latency_s"].quantile(0.95)), 3),
    "priority_values_seen": sorted(df["priority"].dropna().unique().tolist()),
    "sentiment_values_seen": sorted(df["sentiment"].dropna().unique().tolist()),
    "steps_range": (
        int(df["n_repro_steps"].min()) if df["n_repro_steps"].notna().any() else None,
        int(df["n_repro_steps"].max()) if df["n_repro_steps"].notna().any() else None,
    ),
}

print("Overall Summary")
print("=" * 40)
for k, v in overall.items():
    print(f"  {k}: {v}")

# Highlight the key finding
pri_set = set(df["priority"].dropna().unique())
sent_set = set(df["sentiment"].dropna().unique())
if len(pri_set) > 1 or len(sent_set) > 1:
    print()
    print("\u26a0  KEY FINDING: Same customer message, same schema, but models disagree:")
    if len(pri_set) > 1:
        print(f"   Priority: {sorted(pri_set)} \u2192 This changes who gets paged and when.")
    if len(sent_set) > 1:
        print(f"   Sentiment: {sorted(sent_set)} \u2192 This changes the auto-response tone and escalation path.")

Overall Summary
  models_tested: 4
  total_runs: 8
  tier_1_runs: 8
  tier_2_runs: 0
  tier_3_runs: 0
  all_failed_runs: 0
  avg_latency_s: 5.763
  p95_latency_s: 6.958
  priority_values_seen: ['p0', 'p1']
  sentiment_values_seen: ['angry', 'frustrated']
  steps_range: (3, 7)

⚠  KEY FINDING: Same customer message, same schema, but models disagree:
   Priority: ['p0', 'p1'] → This changes who gets paged and when.
   Sentiment: ['angry', 'frustrated'] → This changes the auto-response tone and escalation path.
